# Klasifikasi Dataset CAD Alizadeh — Multi Layer Perceptron (MLP)
**Nama:** Muflih Rafif  
**NIM:** 103052400075  
**Tugas:** Partisipatif 4

Notebook ini melanjutkan **Partisipatif 3** yang telah menggunakan Naive Bayes dan Random Forest.  
Pada Partisipatif 4 ini kita akan:
1. Melakukan klasifikasi menggunakan **Multi Layer Perceptron (MLP)**
2. Melakukan **Hyperparameter Tuning** dengan GridSearchCV
3. **Membandingkan** hasilnya dengan Naive Bayes dan Random Forest dari Partisipatif 3

## 1. Import Library

Library yang digunakan:
- `pandas` & `numpy` — manipulasi data
- `matplotlib` & `seaborn` — visualisasi
- `sklearn.neural_network.MLPClassifier` — implementasi MLP
- `sklearn.model_selection.GridSearchCV` — hyperparameter tuning
- `sklearn.preprocessing.StandardScaler` — normalisasi fitur (wajib untuk MLP)
- `sklearn.pipeline.Pipeline` — menyatukan scaler dan model dalam satu alur
- Library Naive Bayes & Random Forest dari Partisipatif 3 untuk perbandingan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

print("✅ Semua library berhasil diimport.")

## 2. Load & Eksplorasi Data

Dataset **CAD Alizadeh** dibaca dari file Excel.  
Kolom target dideteksi secara otomatis dari kandidat nama umum (`Class`, `class`, `target`, dll.).  
Kita tampilkan 5 baris pertama, ukuran data, dan ringkasan statistik.

In [ ]:
df = pd.read_excel('CADalizadeh.xls')

print(f"Ukuran dataset: {df.shape[0]} baris x {df.shape[1]} kolom")
print("\n5 Baris Pertama:")
display(df.head())

# Deteksi kolom target secara otomatis
candidate_targets = ["Class", "class", "target", "label", "diagnosis", "Class Label"]
target_col = next((c for c in candidate_targets if c in df.columns), df.columns[-1])
print(f"\n✅ Kolom target yang terdeteksi: '{target_col}'")
print(f"Distribusi kelas:\n{df[target_col].value_counts()}")

## 2.1 Visualisasi Awal Data

Menampilkan:
- **Distribusi kelas target** — untuk melihat apakah data seimbang atau tidak
- **Missing value** — kolom mana yang memiliki nilai kosong

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot distribusi kelas
kelas_counts = df[target_col].value_counts()
axes[0].bar(kelas_counts.index.astype(str), kelas_counts.values,
            color=sns.color_palette('Set2', len(kelas_counts)))
axes[0].set_title(f'Distribusi Kelas Target ({target_col})', fontsize=13)
axes[0].set_xlabel('Kelas')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(kelas_counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Plot missing value
missing = df.isna().sum()
missing = missing[missing > 0]
if missing.empty:
    axes[1].text(0.5, 0.5, 'Tidak ada missing value', ha='center',
                 va='center', fontsize=12, transform=axes[1].transAxes)
    axes[1].set_axis_off()
else:
    axes[1].bar(missing.index, missing.values, color='salmon')
    axes[1].set_title('Missing Value per Kolom', fontsize=13)
    axes[1].set_xlabel('Kolom')
    axes[1].set_ylabel('Jumlah Missing')
    axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Preprocessing Data

Langkah preprocessing yang dilakukan:
1. **Drop baris** dengan target kosong
2. **Pisahkan fitur (X) dan target (y)**
3. **Ganti simbol `?`** yang sering dipakai sebagai penanda missing value
4. **Imputasi missing value** — numerik → median, kategorikal → modus
5. **One-Hot Encoding** — mengubah kolom kategorikal menjadi numerik
6. **Split data** — 80% train, 20% test (stratified)
7. **StandardScaler** — WAJIB untuk MLP agar setiap fitur berada pada skala yang sama; difit hanya pada data train untuk menghindari data leakage

In [ ]:
# (1) Drop baris dengan target kosong
df_clean = df.dropna(subset=[target_col]).copy()

# (2) Pisahkan X dan y
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

# (3) Ganti '?' dengan NaN
X = X.replace('?', np.nan)

# (4) Imputasi missing value
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = X[col].fillna(X[col].mode(dropna=True)[0])
    else:
        X[col] = X[col].fillna(X[col].median())

# (5) One-Hot Encoding untuk kolom kategorikal
X = pd.get_dummies(X, drop_first=True)

# (6) Split data train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# (7) StandardScaler — fit HANYA pada train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Jumlah data latih  : {X_train.shape[0]}")
print(f"Jumlah data uji    : {X_test.shape[0]}")
print(f"Jumlah fitur       : {X_train.shape[1]}")
print(f"Distribusi kelas train:\n{y_train.value_counts()}")

## 4. Klasifikasi MLP — Model Dasar (Baseline)

Sebelum hyperparameter tuning, kita latih **MLP Baseline** dengan konfigurasi default sebagai patokan awal.  
Parameter yang digunakan:
- `hidden_layer_sizes=(100,)` — satu hidden layer dengan 100 neuron
- `activation='relu'` — fungsi aktivasi ReLU
- `solver='adam'` — optimizer Adam
- `max_iter=300` — maksimum iterasi training
- `random_state=42` — agar hasil reproducible

In [ ]:
mlp_base = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42
)

mlp_base.fit(X_train_scaled, y_train)
y_pred_mlp_base = mlp_base.predict(X_test_scaled)

# Cross-validation pada data train (5-fold)
cv_scores_base = cross_val_score(
    mlp_base, X_train_scaled, y_train, cv=5, scoring='f1_weighted'
)

print("=== MLP Baseline ===")
print(f"Jumlah iterasi konvergen : {mlp_base.n_iter_}")
print(f"CV F1 (5-fold) mean±std  : {cv_scores_base.mean():.4f} ± {cv_scores_base.std():.4f}")
print(f"Akurasi pada test set    : {accuracy_score(y_test, y_pred_mlp_base):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_mlp_base, zero_division=0))

## 5. Hyperparameter Tuning MLP dengan GridSearchCV

**Hyperparameter tuning** adalah proses mencari kombinasi parameter terbaik untuk sebuah model.  
Kita menggunakan **GridSearchCV** yang mencoba semua kombinasi parameter secara sistematis.

Parameter yang di-tune:
| Parameter | Pilihan | Keterangan |
|---|---|---|
| `hidden_layer_sizes` | (50,), (100,), (100,50), (100,100) | Arsitektur jaringan |
| `activation` | relu, tanh | Fungsi aktivasi |
| `solver` | adam, sgd | Algoritma optimasi |
| `alpha` | 0.0001, 0.001, 0.01 | Regularisasi L2 |
| `learning_rate` | constant, adaptive | Strategi learning rate |

Evaluasi menggunakan **5-fold cross-validation** dengan metrik **F1-Score weighted**.

In [ ]:
param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (100, 50), (100, 100)],
    'activation'        : ['relu', 'tanh'],
    'solver'            : ['adam', 'sgd'],
    'alpha'             : [0.0001, 0.001, 0.01],
    'learning_rate'     : ['constant', 'adaptive'],
}

mlp_gs = MLPClassifier(max_iter=500, random_state=42)

grid_search = GridSearchCV(
    estimator=mlp_gs,
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,          # Gunakan semua core CPU
    verbose=1
)

print("⏳ Proses GridSearchCV sedang berjalan... (harap tunggu)")
grid_search.fit(X_train_scaled, y_train)

print("\n✅ GridSearchCV selesai!")
print(f"Best CV F1-Score  : {grid_search.best_score_:.4f}")
print(f"Best Parameters   :")
for k, v in grid_search.best_params_.items():
    print(f"  {k:25s}: {v}")

## 6. Evaluasi MLP Terbaik (Setelah Tuning)

Model dengan parameter terbaik hasil GridSearchCV dievaluasi pada **data test** yang belum pernah dilihat model sebelumnya.  
Ini memberikan estimasi performa yang tidak bias.

In [ ]:
best_mlp = grid_search.best_estimator_
y_pred_mlp_best = best_mlp.predict(X_test_scaled)

print("=== MLP Terbaik (setelah Hyperparameter Tuning) ===")
print(f"Jumlah iterasi   : {best_mlp.n_iter_}")
print(f"Akurasi test set : {accuracy_score(y_test, y_pred_mlp_best):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_mlp_best, zero_division=0))

# Confusion Matrix MLP terbaik
fig, ax = plt.subplots(figsize=(6, 5))
cm_mlp = confusion_matrix(y_test, y_pred_mlp_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_mlp,
                               display_labels=best_mlp.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — MLP Terbaik', fontsize=13)
plt.tight_layout()
plt.show()

## 6.1 Visualisasi Proses Tuning

Grafik berikut menampilkan **Top 20 kombinasi parameter** terbaik berdasarkan CV F1-Score,  
sehingga kita dapat melihat rentang performa yang dicapai selama tuning.

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)
top20 = cv_results.nlargest(20, 'mean_test_score').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(top20))]
bars = ax.barh(range(len(top20)), top20['mean_test_score'], xerr=top20['std_test_score'],
               color=colors, capsize=4, edgecolor='white')
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([f"Rank {i+1}" for i in range(len(top20))], fontsize=9)
ax.set_xlabel('Mean CV F1-Score (weighted)', fontsize=11)
ax.set_title('Top 20 Kombinasi Hyperparameter (GridSearchCV)', fontsize=13)
ax.axvline(grid_search.best_score_, color='red', linestyle='--', label=f'Best: {grid_search.best_score_:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

print("Top 5 Kombinasi Terbaik:")
cols_show = ['mean_test_score', 'std_test_score',
             'param_hidden_layer_sizes', 'param_activation',
             'param_solver', 'param_alpha', 'param_learning_rate']
display(top20[cols_show].head())

## 7. Perbandingan Semua Model

Pada tahap ini kita melatih ulang **Naive Bayes** dan **Random Forest** (sama persis dengan Partisipatif 3)  
pada data yang identik, lalu membandingkan performa keempat model:

| Model | Keterangan |
|---|---|
| Naive Bayes | Dari Partisipatif 3 |
| Random Forest | Dari Partisipatif 3 |
| MLP Baseline | MLP dengan parameter default |
| MLP Tuned | MLP setelah GridSearchCV |

**Catatan:** Naive Bayes & Random Forest tidak membutuhkan data yang sudah di-scale, sehingga menggunakan `X_train` / `X_test` tanpa StandardScaler.

In [ ]:
# ── Naive Bayes (Partisipatif 3) ──
model_nb = GaussianNB()
model_nb.fit(X_train, y_train)
y_pred_nb = model_nb.predict(X_test)

# ── Random Forest (Partisipatif 3) ──
model_rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

print("✅ Naive Bayes & Random Forest selesai dilatih.")
print("✅ MLP Baseline & MLP Tuned sudah tersedia dari langkah sebelumnya.")

In [ ]:
def hitung_metrik(y_true, y_pred, nama_model):
    return {
        'Model'    : nama_model,
        'Accuracy' : round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 4),
        'Recall'   : round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 4),
        'F1-Score' : round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 4),
    }

hasil_semua = [
    hitung_metrik(y_test, y_pred_nb,       'Naive Bayes        (P3)'),
    hitung_metrik(y_test, y_pred_rf,       'Random Forest      (P3)'),
    hitung_metrik(y_test, y_pred_mlp_base, 'MLP Baseline       (P4)'),
    hitung_metrik(y_test, y_pred_mlp_best, 'MLP Tuned          (P4)'),
]

df_hasil = pd.DataFrame(hasil_semua).set_index('Model')

print("=" * 65)
print("  TABEL PERBANDINGAN PERFORMA — PARTISIPATIF 3 vs PARTISIPATIF 4")
print("=" * 65)
display(df_hasil)

model_terbaik = df_hasil['F1-Score'].idxmax()
print(f"\n🏆 Model terbaik berdasarkan F1-Score: {model_terbaik.strip()}")

## 7.1 Visualisasi Perbandingan Performa

Grafik batang berkelompok memudahkan kita melihat secara visual perbedaan performa  
keempat model pada keempat metrik evaluasi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Grouped bar chart — semua metrik
df_plot = df_hasil.copy()
df_plot.index = [idx.strip() for idx in df_plot.index]  # strip whitespace untuk label
x = np.arange(len(df_plot))
width = 0.2
metriks = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors  = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

for i, (metrik, color) in enumerate(zip(metriks, colors)):
    axes[0].bar(x + i*width, df_plot[metrik], width, label=metrik, color=color, alpha=0.85)

axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(df_plot.index, fontsize=9, rotation=10)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel('Score')
axes[0].set_title('Perbandingan Semua Metrik per Model', fontsize=12)
axes[0].legend(loc='lower right')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

for i, (metrik, color) in enumerate(zip(metriks, colors)):
    for j, val in enumerate(df_plot[metrik]):
        axes[0].text(j + i*width, val + 0.01, f'{val:.3f}', ha='center',
                     va='bottom', fontsize=7, fontweight='bold')

# (b) F1-Score comparison only
f1_vals = df_plot['F1-Score']
bar_colors = ['#C44E52' if v == f1_vals.max() else '#4C72B0' for v in f1_vals]
axes[1].bar(df_plot.index, f1_vals, color=bar_colors, edgecolor='white', width=0.5)
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('F1-Score')
axes[1].set_title('Perbandingan F1-Score (Merah = Terbaik)', fontsize=12)
axes[1].tick_params(axis='x', rotation=12)
axes[1].grid(axis='y', linestyle='--', alpha=0.5)
for i, v in enumerate(f1_vals):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## 7.2 Confusion Matrix Semua Model

Confusion Matrix menampilkan rincian prediksi benar dan salah per kelas untuk masing-masing model.

In [ ]:
models_info = [
    ('Naive Bayes (P3)',        y_pred_nb,       None),
    ('Random Forest (P3)',      y_pred_rf,        None),
    ('MLP Baseline (P4)',       y_pred_mlp_base,  best_mlp.classes_),
    ('MLP Tuned (P4)',          y_pred_mlp_best,  best_mlp.classes_),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, (nama, y_pred, classes) in enumerate(models_info):
    cm = confusion_matrix(y_test, y_pred)
    if classes is None:
        labels = sorted(y_test.unique())
    else:
        labels = classes
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    axes[i].set_title(f'{nama}\nF1={f1:.4f}', fontsize=11)

plt.suptitle('Confusion Matrix — Perbandingan Semua Model', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Kesimpulan

### Ringkasan Proses

| Langkah | Keterangan |
|---|---|
| Load Data | Dataset CAD Alizadeh dibaca dari Excel |
| Preprocessing | Imputasi missing value, One-Hot Encoding, StandardScaler |
| MLP Baseline | Satu hidden layer (100 neuron), parameter default |
| GridSearchCV | 5-fold CV, cari kombinasi terbaik dari 96 kemungkinan |
| MLP Terbaik | Model dengan F1-CV terbaik dievaluasi di test set |
| Perbandingan | Dibandingkan dengan Naive Bayes & Random Forest (P3) |

### Analisis Perbandingan

1. **Naive Bayes** — Model sederhana yang berasumsi independensi antar fitur. Performa cenderung lebih rendah karena asumsi tersebut sering tidak terpenuhi pada data medis.

2. **Random Forest** — Model ensemble yang kuat, tidak memerlukan scaling, dan mampu menangani fitur yang berkorelasi. Umumnya memberikan performa yang baik.

3. **MLP Baseline** — Sudah mampu menangkap pola non-linear berkat hidden layer, namun belum dioptimalkan.

4. **MLP Tuned** — Setelah GridSearchCV, performa MLP meningkat karena arsitektur dan hyperparameter disesuaikan dengan karakteristik data CAD Alizadeh.

### Kelebihan MLP dibanding Metode Sebelumnya
- Mampu memodelkan **hubungan non-linear** yang kompleks antar fitur
- Dengan jumlah hidden layer dan neuron yang tepat, MLP dapat merepresentasikan fungsi klasifikasi yang lebih ekspresif
- Hyperparameter tuning memungkinkan model disesuaikan secara sistematis dengan data

### Rekomendasi
Model dengan **F1-Score tertinggi** pada tabel perbandingan adalah pilihan terbaik untuk klasifikasi CAD Alizadeh.  
Untuk deployment, pertimbangkan juga **waktu inferensi** dan **interpretabilitas** model sesuai kebutuhan klinis.